# Spark Bronze wikpedia page reads

In [4]:
exeuction_date = "2025-01-01"
full_refresh = True

In [5]:

import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg


def get_spark_session():
    """Get spark client for s3."""
    s3_cfg = get_config("s3")
    spark_cfg = get_config("spark")

    print(f"using s3 endpoint: {s3_cfg['url']}")
    print(f"using spark master: {spark_cfg['master_url']}")

    spark_session = (  SparkSession
        .builder
        .master(spark_cfg['master_url'])
        .appName("Wikipedia page reads - Bronze")
        .config("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .config("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
        .config("spark.hadoop.fs.s3a.endpoint", s3_cfg["url"])
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.sql.warehouse.dir", "s3a://dwh/warehouse/")
        .config(
            "spark.jars",
            "../jars/hadoop-aws-3.3.4.jar,../jars/aws-java-sdk-bundle-1.12.262.jar")
        .config("spark.executor.memory", "2g")
        .getOrCreate()
    )

    return spark_session


## Bronze
Performs: 
* ingestion
* column naming
* column casting
* get the date from the filename
* partitioning

In [8]:
from pyspark.sql.functions import input_file_name, col, sum as _sum, substring, to_date

spark = get_spark_session()

s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/**/*.gz"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-200000.gz"


schema = StructType([
    StructField(name="domain_code", dataType=StringType(), nullable = True),
    StructField("page_title", StringType(), True),
    StructField("count_views", StringType(), True),
    # StructField("total_response_size", StringType(), True), this eems not to be populated, discard it
])
spark.sparkContext.setLogLevel("WARN")
print(f"Reading data from {s3_path}")

df_base = (
    spark.read.format("csv")
    .option("delimiter", " ")
    .option("header", "false")
    .option("inferSchema", "false")
    .schema(schema)
    .load(s3_path)
)

df_base = (
    df_base
    .na.drop(subset=["domain_code"])
    .filter(~col("page_title").contains(":"))
    .filter(~col("page_title").isin("-", 'Main_Page', 'Forside', 'Hauptseite', 'wiki.phtml'))
    .withColumn("country_code", substring(col("domain_code"), 1, 2))
    .filter(col("domain_code").isin("sv", 'dk', 'no', 'de', 'en'))
    .withColumn("count_views", col("count_views").cast(IntegerType()))
    .withColumn("file_name", input_file_name())
    .withColumn("date", to_date(substring(col("file_name"), -18, 8), 'yyyyMMdd'))
)
# .repartition("date").cache()
spark.sql("DROP DATABASE IF EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("DROP TABLE IF EXISTS bronze.wikipedia_page_reads ")

(   df_base
    .write
    .mode("overwrite")   # Options: 'overwrite', 'append', 'ignore', 'error' (default)
    .format("parquet")    # Options: 'parquet', 'csv', 'json', 'orc', etc.
    .partitionBy("date")
    .saveAsTable("bronze.wikipedia_page_reads")
)

retrieving s3 config from http://tfds-config:8005/api/configs/s3
retrieving spark config from http://tfds-config:8005/api/configs/spark
using s3 endpoint: http://s3-minio:9000
using spark master: spark://spark-master:7077
Reading data from s3a://data/wikipedia_pageviews/2025/2025-03/11/pageviews-20250311-200000.gz


25/04/12 14:01:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [9]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

aggregated_df = (
    spark.table("bronze.wikipedia_page_reads")
    .groupBy('date', "page_title", "country_code")
    .agg(
        _sum(col("count_views")).alias("total_count_views"),
    )
)

national_win = (
    Window
    .partitionBy('date', "country_code")
    .orderBy(col("total_count_views").desc())
)

ranked_df = (
    aggregated_df
    .withColumn("national_rank", row_number().over(national_win))
    )

final_df = (
    ranked_df
    .filter(col('national_rank') <= 3)
    .orderBy('date', "national_rank", "country_code")
)

# final_df.explain(mode="extended")

In [10]:
final_df.show(50, truncate=False)

+----------+---------------------+------------+-----------------+-------------+
|date      |page_title           |country_code|total_count_views|national_rank|
+----------+---------------------+------------+-----------------+-------------+
|2025-03-11|Elon_Musk            |de          |1953             |1            |
|2025-03-11|Apple_Network_Server |en          |5470             |1            |
|2025-03-11|Kvener               |no          |68               |1            |
|2025-03-11|Jan_Stenbeck         |sv          |322              |1            |
|2025-03-11|Martin_Eberhard      |de          |985              |2            |
|2025-03-11|Biggest_ball_of_twine|en          |4901             |2            |
|2025-03-11|Tornedalen           |no          |64               |2            |
|2025-03-11|Margaretha_af_Ugglas |sv          |205              |2            |
|2025-03-11|Kalt_ist_die_Angst   |de          |872              |3            |
|2025-03-11|Limonene             |en    

In [ ]:
# ranked_df.explain(mode="extended")
# spark.stop()